In [1]:
#
# Preparacion de datos para el LAB:
#
# https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients?utm_source=ibm_developer&utm_content=in_content_link&utm_id=tutorials_awb-classifying-data-svm-algorithm-python
#
# """
#
# import pandas as pd  # type : ignore
#
# df = pd.read_csv("../files/input/default_of_credit_card_clients.csv")
# train = df.sample(frac=0.7, random_state=1)
# test = df.drop(train.index)
#
# train.to_csv(
#     "../files/input/train_default_of_credit_card_clients.csv.zip",
#     index=False,
#     compression="zip",
# )
# test.to_csv(
#     "../files/input/test_default_of_credit_card_clients.csv.zip",
#     index=False,
#     compression="zip",
# )

In [2]:
#
# En este dataset se desea pronosticar el default (pago) del cliente el próximo
# mes a partir de 23 variables explicativas.
#
#   LIMIT_BAL: Monto del credito otorgado. Incluye el credito individual y el
#              credito familiar (suplementario).
#         SEX: Genero (1=male; 2=female).
#   EDUCATION: Educacion (0=N/A; 1=graduate school; 2=university; 3=high school; 4=others).
#    MARRIAGE: Estado civil (0=N/A; 1=married; 2=single; 3=others).
#         AGE: Edad (years).
#       PAY_0: Historia de pagos pasados. Estado del pago en septiembre, 2005.
#       PAY_2: Historia de pagos pasados. Estado del pago en agosto, 2005.
#       PAY_3: Historia de pagos pasados. Estado del pago en julio, 2005.
#       PAY_4: Historia de pagos pasados. Estado del pago en junio, 2005.
#       PAY_5: Historia de pagos pasados. Estado del pago en mayo, 2005.
#       PAY_6: Historia de pagos pasados. Estado del pago en abril, 2005.
#   BILL_AMT1: Historia de pagos pasados. Monto a pagar en septiembre, 2005.
#   BILL_AMT2: Historia de pagos pasados. Monto a pagar en agosto, 2005.
#   BILL_AMT3: Historia de pagos pasados. Monto a pagar en julio, 2005.
#   BILL_AMT4: Historia de pagos pasados. Monto a pagar en junio, 2005.
#   BILL_AMT5: Historia de pagos pasados. Monto a pagar en mayo, 2005.
#   BILL_AMT6: Historia de pagos pasados. Monto a pagar en abril, 2005.
#    PAY_AMT1: Historia de pagos pasados. Monto pagado en septiembre, 2005.
#    PAY_AMT2: Historia de pagos pasados. Monto pagado en agosto, 2005.
#    PAY_AMT3: Historia de pagos pasados. Monto pagado en julio, 2005.
#    PAY_AMT4: Historia de pagos pasados. Monto pagado en junio, 2005.
#    PAY_AMT5: Historia de pagos pasados. Monto pagado en mayo, 2005.
#    PAY_AMT6: Historia de pagos pasados. Monto pagado en abril, 2005.
#
# La variable "default payment next month" corresponde a la variable objetivo.
#
# El dataset ya se encuentra dividido en conjuntos de entrenamiento y prueba
# en la carpeta "files/input/".
#
# Los pasos que debe seguir para la construcción de un modelo de
# clasificación están descritos a continuación.
#
import pandas as pd  #  type: ignore

raw_train_df = pd.read_csv(
    "../files/input/train_data.csv.zip",
    compression="zip",
)

raw_test_df = pd.read_csv(
    "../files/input/test_data.csv.zip",
    compression="zip",
)

display(raw_train_df.head())
display(raw_test_df.head())

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,10748,310000,1,3,1,32,0,0,0,0,...,84373,57779,14163,8295,6000,4000,3000,1000,2000,0
1,12574,10000,2,3,1,49,-1,-1,-2,-1,...,1690,1138,930,0,0,2828,0,182,0,1
2,29677,50000,1,2,1,28,-1,-1,-1,0,...,45975,1300,43987,0,46257,2200,1300,43987,1386,0
3,8857,80000,2,3,1,52,2,2,3,3,...,40748,39816,40607,3700,1600,1600,0,1600,1600,1
4,21099,270000,1,1,2,34,1,2,0,0,...,22448,15490,17343,0,4000,2000,0,2000,2000,0


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
1,10,20000,1,3,2,35,-2,-2,-2,-2,...,0,13007,13912,0,0,0,13007,1122,0,0
2,11,200000,2,3,2,34,0,0,2,0,...,2513,1828,3731,2306,12,50,300,3738,66,0
3,15,250000,1,1,2,29,0,0,0,0,...,59696,56875,55512,3000,3000,3000,3000,3000,3000,0
4,16,50000,2,3,3,23,1,2,0,0,...,28771,29531,30211,0,1500,1100,1200,1300,1100,0


In [3]:
## from pprint import pprint
##
## pprint(raw_train_df.columns)

In [4]:
## # def check_data(df):
## #     display(df.SEX.value_counts().sort_index())
## #     print()
## #     display(df.EDUCATION.value_counts().sort_index())
## #     print()
## #     display(df.MARRIAGE.value_counts().sort_index())
## #     print()
## #     display(df.AGE.value_counts().sort_index())
## #     print()
## #     display(df.isnull().sum().sort_index())
##
##
## #        SEX: Genero (1=male; 2=female).
## #  EDUCATION: Educacion (1=graduate school; 2=university; 3=high school; 4=others).
## #   MARRIAGE: Estado civil (1=married; 2=single; 3=others).
## #        AGE: Edad (years).
## check_data(raw_train_df)

In [5]:
## #        SEX: Genero (1=male; 2=female).
## #  EDUCATION: Educacion (1=graduate school; 2=university; 3=high school; 4=others).
## #   MARRIAGE: Estado civil (1=married; 2=single; 3=others).
## #        AGE: Edad (years).
## check_data(raw_test_df)

In [6]:
#
# Paso 1.
# Realice la limpieza de los datasets:
# - Renombre la columna "default payment next month" a "default".
# - Remueva la columna "ID".
# - Elimine los registros con informacion no disponible.
# - Para la columna EDUCATION, valores > 4 indican niveles superiores
#   de educación, agrupe estos valores en la categoría "others".
# - Renombre la columna "default payment next month" a "default"
# - Remueva la columna "ID".
#
def preprocess_data(df):
    df = df.copy()
    df = df.rename(columns={"default payment next month": "default"})
    df = df.drop(columns=["ID"])
    df = df.loc[(df["EDUCATION"] != 0)]
    df = df.loc[(df["MARRIAGE"] != 0)]
    df.loc[df["EDUCATION"] > 4, "EDUCATION"] = 4
    return df


cleaned_train_df = preprocess_data(raw_train_df)
cleaned_test_df = preprocess_data(raw_test_df)

In [7]:
#
# Paso 2.
# Divida los datasets en x_train, y_train, x_test, y_test.
#
import os
import pickle

x_train = cleaned_train_df.drop(columns=["default"])
y_train = cleaned_train_df["default"]

x_test = cleaned_test_df.drop(columns=["default"])
y_test = cleaned_test_df["default"]

if not os.path.exists("../files/grading/"):
    os.makedirs("../files/grading/")

with open("../files/grading/x_train.pkl", "wb") as f:
    pickle.dump(x_train, f)

with open("../files/grading/y_train.pkl", "wb") as f:
    pickle.dump(y_train, f)

with open("../files/grading/x_test.pkl", "wb") as f:
    pickle.dump(x_test, f)

with open("../files/grading/y_test.pkl", "wb") as f:
    pickle.dump(y_test, f)

In [8]:
#
# Paso 3.
# Cree un pipeline para el modelo de clasificación. Este pipeline debe
# contener las siguientes capas:
# - Transforma las variables categoricas usando el método
#   one-hot-encoding.
# - Ajusta un modelo de bosques aleatorios (rando forest).
#
from sklearn.compose import ColumnTransformer  # type: ignore
from sklearn.ensemble import RandomForestClassifier  # type: ignore
from sklearn.decomposition import PCA  # type: ignore
from sklearn.pipeline import Pipeline  # type: ignore
from sklearn.preprocessing import OneHotEncoder  # type: ignore
from sklearn.feature_selection import f_classif  # type: ignore


# Columnas categoricas:
#        SEX: Genero (1=male; 2=female).
#  EDUCATION: Educacion (0=N/A; 1=graduate school; 2=university; 3=high school; 4=others).
#   MARRIAGE: Estado civil (0=N/A; 1=married; 2=single; 3=others).
pipeline = Pipeline(
    [
        (
            "transformer",
            ColumnTransformer(
                [
                    (
                        "encoder",
                        OneHotEncoder(),
                        ["SEX", "EDUCATION", "MARRIAGE"],
                    ),
                ],
                remainder="passthrough",
            ),
        ),
        ("model", RandomForestClassifier()),
    ]
)

In [9]:
#
# Paso 4.
# Optimice los hiperparametros del pipeline usando validación cruzada.
# Use 10 splits para la validación cruzada. Use la función de precision
# balanceada para medir la precisión del modelo.
#
import warnings
from sklearn.model_selection import GridSearchCV  # type: ignore

warnings.filterwarnings("ignore")

param_grid = {
    "model__n_estimators": [200],
    "model__max_features": ["sqrt"],
    "model__max_depth": [None],
    "model__min_samples_split": [11],
    "model__min_samples_leaf": [2],
}

grid_search_pipeline = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=10,
    scoring="balanced_accuracy",
    n_jobs=-1,
)

grid_search_pipeline.fit(x_train, y_train)

print(grid_search_pipeline.best_estimator_)
print(grid_search_pipeline.score(x_train, y_train))
print(grid_search_pipeline.score(x_test, y_test))

Pipeline(steps=[('transformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('encoder', OneHotEncoder(),
                                                  ['SEX', 'EDUCATION',
                                                   'MARRIAGE'])])),
                ('model',
                 RandomForestClassifier(min_samples_leaf=2,
                                        min_samples_split=11,
                                        n_estimators=200))])
0.7861947473372171
0.6730212983503322


In [10]:
#
# Paso 5.
# Guarde el modelo como "files/models/model.pkl".
#
import os
import pickle

if not os.path.exists("../files/models"):
    os.makedirs("../files/models")

with open("../files/models/model.pkl", "wb") as file:
    pickle.dump(grid_search_pipeline, file)

In [11]:
#
# Paso 6.
# Calcule las metricas de precision, precision balanceada, recall,
# y f1-score para los conjuntos de entrenamiento y prueba.
# Guardelas en el archivo files/output/metrics.json. Cada fila
# del archivo es un diccionario con las metricas de un modelo.
# Este diccionario tiene un campo para indicar si es el conjunto
# de entrenamiento o prueba. Por ejemplo:
#
# {'dataset': 'train', 'precision': 0.8, 'balanced_accuracy': 0.7, 'recall': 0.9, 'f1_score': 0.85}
# {'dataset': 'test', 'precision': 0.7, 'balanced_accuracy': 0.6, 'recall': 0.8, 'f1_score': 0.75}
#

import os

from sklearn.metrics import precision_score, balanced_accuracy_score, recall_score, f1_score  # type: ignore

if not os.path.exists("../files/output"):
    os.makedirs("../files/output")

metrics = {
    "type": "metrics",
    "dataset": "train",
    "precision": float(precision_score(y_train, grid_search_pipeline.predict(x_train))),
    "balanced_accuracy": float(
        balanced_accuracy_score(y_train, grid_search_pipeline.predict(x_train))
    ),
    "recall": float(recall_score(y_train, grid_search_pipeline.predict(x_train))),
    "f1_score": float(f1_score(y_train, grid_search_pipeline.predict(x_train))),
}

with open("../files/output/metrics.json", "w") as file:
    file.write(str(metrics).replace("'", '"'))
    file.write("\n")

display(metrics)

metrics = {
    "type": "metrics",
    "dataset": "test",
    "precision": float(precision_score(y_test, grid_search_pipeline.predict(x_test))),
    "balanced_accuracy": float(
        balanced_accuracy_score(y_test, grid_search_pipeline.predict(x_test))
    ),
    "recall": float(recall_score(y_test, grid_search_pipeline.predict(x_test))),
    "f1_score": float(f1_score(y_test, grid_search_pipeline.predict(x_test))),
}

with open("../files/output/metrics.json", "a") as file:
    file.write(str(metrics).replace("'", '"'))
    file.write("\n")

display(metrics)

{'type': 'metrics',
 'dataset': 'train',
 'precision': 0.944082332761578,
 'balanced_accuracy': 0.7861947473372171,
 'recall': 0.5824338624338624,
 'f1_score': 0.7204188481675393}

{'type': 'metrics',
 'dataset': 'test',
 'precision': 0.6597760551248923,
 'balanced_accuracy': 0.6730212983503322,
 'recall': 0.4018887722980063,
 'f1_score': 0.4995109227257907}

In [12]:
#
# Paso 7.
# Calcule las matrices de confusion para los conjuntos de entrenamiento y
# prueba. Guardelas en el archivo files/output/metrics.json. Cada fila
# del archivo es un diccionario con las metricas de un modelo.
# de entrenamiento o prueba. Por ejemplo:
#
# {'type': 'cm_matrix', 'dataset': 'train', 'true_0': {"predicted_0": 15562, "predicte_1": 666}, 'true_1': {"predicted_0": 3333, "predicted_1": 1444}}
# {'type': 'cm_matrix', 'dataset': 'test', 'true_0': {"predicted_0": 15562, "predicte_1": 650}, 'true_1': {"predicted_0": 2490, "predicted_1": 1420}}
#
from sklearn.metrics import confusion_matrix


cm_train = pd.DataFrame(
    data=confusion_matrix(y_train, grid_search_pipeline.predict(x_train)),
    index=["True 0", "True 1"],
    columns=["Predicted 0", "Predicted 1"],
)

metrics = {
    "type": "cm_matrix",
    "dataset": "train",
    "true_0": {
        "predicted_0": int(cm_train.loc["True 0", "Predicted 0"]),
        "predicted_1": int(cm_train.loc["True 0", "Predicted 1"]),
    },
    "true_1": {
        "predicted_0": int(cm_train.loc["True 1", "Predicted 0"]),
        "predicted_1": int(cm_train.loc["True 1", "Predicted 1"]),
    },
}

with open("../files/output/metrics.json", "a") as file:
    file.write(str(metrics).replace("'", '"'))
    file.write("\n")

display(metrics)

cm_test = pd.DataFrame(
    data=confusion_matrix(y_test, grid_search_pipeline.predict(x_test)),
    index=["True 0", "True 1"],
    columns=["Predicted 0", "Predicted 1"],
)

metrics = {
    "type": "cm_matrix",
    "dataset": "test",
    "true_0": {
        "predicted_0": int(cm_test.loc["True 0", "Predicted 0"]),
        "predicted_1": int(cm_test.loc["True 0", "Predicted 1"]),
    },
    "true_1": {
        "predicted_0": int(cm_test.loc["True 1", "Predicted 0"]),
        "predicted_1": int(cm_test.loc["True 1", "Predicted 1"]),
    },
}

with open("../files/output/metrics.json", "a") as file:
    file.write(str(metrics).replace("'", '"'))
    file.write("\n")

display(metrics)

{'type': 'cm_matrix',
 'dataset': 'train',
 'true_0': {'predicted_0': 16065, 'predicted_1': 163},
 'true_1': {'predicted_0': 1973, 'predicted_1': 2752}}

{'type': 'cm_matrix',
 'dataset': 'test',
 'true_0': {'predicted_0': 6678, 'predicted_1': 395},
 'true_1': {'predicted_0': 1140, 'predicted_1': 766}}